## TODO
- utilize tags to search prompts which is not supported open-sourced mlflow

In [15]:
import orjson
## CANNOT import, support Databricks(cloud) Runtime ML 14.2+
# import mlflow.genai
import mlflow

In [ ]:
def load_prompt(run_id: str, artifact_path: str = "prompt.txt") -> str:
    import mlflow
    try:
        local_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=artifact_path)
        with open(local_path, "r") as f:
            return f.read()
    except Exception:
        from mlflow.tracking import MlflowClient
        client = MlflowClient()
        run = client.get_run(run_id)
        return run.data.params.get("prompt") or run.data.tags.get("prompt")

def log_prompt(prompt, response, model=None, reference=None) -> list[str]:
    """
    Can log params whatever you want for specific 'run'
    and we can get the 'run' info
    """
    responses = dict()
    with mlflow.start_run() as run:
        responses['run_id'] = run.info.run_id
        result = mlflow.log_param("prompt", prompt)
        responses['log_param_prompt'] = result
        result = mlflow.log_param("response", response)
        responses['log_param_response'] = result
        if model:
            result = mlflow.set_tag("model", model)
            responses['set_tag_model'] = result
        if reference:
            result = mlflow.log_param("reference", reference)
            responses['log_param_references'] = result
    return responses

# def run_and_log(prompt, model="llama3", reference=None):
#     import time
#     start = time.time()

#     res = requests.post(
#         "http://localhost:11434/api/generate",
#         json={"model": model, "prompt": prompt, "stream": False}
#     )
#     output = res.json()["response"]
#     latency = int((time.time() - start) * 1000)

## SUPPORT BY Databricks(cloud) runtime
#     with mlflow.start_run():
#         mlflow.genai.log_prompt(
#             prompt=prompt,
#             response=output,
#             model=f"ollama/{model}",
#             latency_ms=latency,
#             reference=reference,
#             evaluator="bertscore" if reference else None
#         )
#     return output


In [ ]:
from langchain_ollama import ChatOllama

lang_model_name = "tiger-gemma2"
llm = ChatOllama(
    model=lang_model_name,
    temperature=0.8,
    num_predict=2048,
    num_gpu=-1
)

In [5]:
question = "Why is the sky blue?"
response = llm.invoke(question)

## Log prompt and Get run_id

In [36]:
response_log_prompt = log_prompt(prompt=question, response=response,model=lang_model_name)
response_log_prompt

2025/06/28 15:43:26 INFO mlflow.tracking._tracking_service.client: 🏃 View run fearless-bass-990 at: http://0.0.0.0:38080/#/experiments/0/runs/9dad2cbc5c704f6dae8275802146741b.
2025/06/28 15:43:26 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://0.0.0.0:38080/#/experiments/0.


{'run_id': '9dad2cbc5c704f6dae8275802146741b',
 'log_param_prompt': 'Why is the sky blue?',
 'log_param_response': AIMessage(content="The sky appears blue due to a phenomenon called Rayleigh scattering.\n\nSunlight, which appears white to our eyes, is actually composed of all colors of the rainbow. When this sunlight enters Earth's atmosphere, it interacts with tiny air molecules such as nitrogen and oxygen. These molecules are much smaller than the wavelength of visible light.\n\nAs sunlight passes through the atmosphere, its blue wavelengths (which have shorter wavelengths) scatter more effectively than other colors due to Rayleigh scattering. This means that blue light gets redirected in many directions, making the sky appear blue from our perspective on Earth.\n\nAt sunset and sunrise, when the sun is low on the horizon, light has to travel through a thicker layer of atmosphere. During this time, the longer wavelengths (like red and orange) are more likely to be scattered away, lea

In [40]:
run_id_for_prompt = response_log_prompt['run_id']
run_id_for_prompt

'9dad2cbc5c704f6dae8275802146741b'

## Load prompt by run_id

In [41]:
prompt = load_prompt(run_id=run_id_for_prompt)
prompt

'Why is the sky blue?'

## Get run-info by run_id

In [46]:
client = mlflow.MlflowClient()
run: mlflow.entities.run.Run = client.get_run(run_id_for_prompt)
run

<Run: data=<RunData: metrics={}, params={'prompt': 'Why is the sky blue?',
 'response': 'content="The sky appears blue due to a phenomenon called '
             'Rayleigh scattering.\\n\\nSunlight, which appears white to our '
             'eyes, is actually composed of all colors of the rainbow. When '
             "this sunlight enters Earth's atmosphere, it interacts with tiny "
             'air molecules such as nitrogen and oxygen. These molecules are '
             'much smaller than the wavelength of visible light.\\n\\nAs '
             'sunlight passes through the atmosphere, its blue wavelengths '
             '(which have shorter wavelengths) scatter more effectively than '
             'other colors due to Rayleigh scattering. This means that blue '
             'light gets redirected in many directions, making the sky appear '
             'blue from our perspective on Earth.\\n\\nAt sunset and sunrise, '
             'when the sun is low on the horizon, light has to trav